# 轮次

罗马不是一日建成的。同样，神经网络模型也不是一次训练就可以学会的。

在深度学习中，模型训练遵循一个朴素的规则：利用海量数据，通过成千上万次的**反复试错**，以**小步快跑**的方式逐渐逼近最佳模型参数。这种**重复且不断改进**的过程，称为**迭代**（Iteration）。

我们把一组训练数据提供给网络模型学习，称为一次**迭代**。迭代包括三个步骤：

* **前向传播**：
* **方向传播**：
* **参数更新**：

某些模型训练策略可能经过多次的前向传播和反向传播，才会进行一次参数更新。因此准确判断迭代的标准是：每完成一次参数更新，称为一次**迭代**。

### 轮次

顺序或者随机地把所有训练数据都迭代一次，称为称为一个**轮次**（Epoch）。由于学习率的存在，我们在模型训练过程中有意地控制参数更新的幅度，所以通常模型训练都需要经过多个轮次。

轮数（Epochs）是我们遇到的第二个超参数。

In [1]:
import numpy as np

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    def __str__(self):
        return f'Tensor({self.data})'

## 数据集

模型训练所需要的海量数据，称为**数据集**（Dataset）。

数据集的每一条数据，称为一个**样本**（Sample）。每个样本都需要包括特征值和标签值两部分。

---

通常数据集会被随机分成两部分：
* **训练集**（Training Set）：较多的一部分（比如：80%），用于模型训练；
* **测试集**（Test Set）：另一部分较少的（比如：20%），用于模型评估。

划分数据集的目的是为了保留一部分**新数据**：测试集。这样，可以用**新数据**验证训练过的网络模型是否真正学会了规律，还是只死记硬背了一些内容。

这就像是期末考试，老师不会用你做过的练习题做考卷，而是用一套新题来考察你是不是真的掌握了学习的知识。

### 批处理

每次迭代，如果我们用一个样本进行训练，这种方式称为**随机梯度下降**（Stochastic Gradient Descent, SGD）。虽然这种方式简单，但是无法充分利用 NumPy 的矢量并行计算能力。

实际上，我们可以一次将多个样本送入网络模型进行训练。这种方式称为**批处理**（Batch Processing），是现代深度学习的标准做法，可以充分利用 GPU 的性能。

按照每次使用的样本数量，梯度下降可分为三类：

* **随机梯度下降**（SGD）:每次把**一个样本**送入迭代；
* **批量梯度下降**（Batch GD）：每次把**全部样本**送入迭代。实践中很少使用批量梯度下降进行训练，因为需要大量的内存，运算速度会非常缓慢；
* **小批量梯度下降**（Mini-batch GD）：每次把**数个样本**送入迭代。小批量梯度下降充分地利用了 NumPy（或者其他计算库）的矢量并行计算能力，是目前常规的模型训练方式。

---

数据集通常可以通过初始化参数 **batch_size**（批大小）来控制批处理方式：

* batch_size 的缺省值为 1，就是每次只送出一个样本，即**随机梯度下降**；
* 当把 batch_size 设置为全部训练集数据的数量时，就是每次都送出全部样本，即**批量梯度下降**；
* 当把 batch_size 设置为两者之间的其他数值时，比如 2，就是每次送出两个样本，即**小批量梯度下降**。

数据集通常会提供的函数包括：

* **load**（加载函数）：加载全部数据；
* **train**（训练函数）：切换到模型训练模式；
* **eval**（测试函数）：切换到模型测试模式；
* **all**（全部样本）：返回全部样本；
* **len**（样本数量）：返回样本数量；
* **getitem(index)**（单个样本）：根据索引（index）返回对应的样本。

需要注意的是，以上的函数都已经把批处理考虑在内。比如在**批量梯度下降**的状态下，**样本数量**（len）为 1；又或者在**小批量梯度下降**的状态下，**单个样本**（getitem）实际会返回数个（batch_size）样本。

In [3]:
class Dataset:

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.load()
        self.train()

    def load(self):
        self.train_data = ([[22.5, 72.0],
                            [31.4, 45.0],
                            [19.8, 85.0],
                            [27.6, 63.0]],
                           [[95],
                            [210],
                            [70],
                            [155]])
        self.test_data = ([[28.1, 58.0]],
                          [[165]])

    def train(self):
        self.data = self.train_data

    def eval(self):
        self.data = self.test_data

    def all(self):
        x, y = self.data
        return Tensor(x), Tensor(y)

    def __len__(self):
        x, *_ = self.data
        return len(x) // self.batch_size

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y = self.data
        return Tensor(x[s]), Tensor(y[s])

## 模型

In [4]:
class Linear:

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def __call__(self, x: Tensor):
        return self.forward(x)

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)

        p.gradient_fn = gradient_fn
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

In [5]:
class MSELoss:

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / y.data.size

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

In [6]:
class SGDOptimizer:

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

In [7]:
class NNModel:

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    def train(self, dataset, epochs):
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction = self.layer(feature)
                loss = self.loss_fn(prediction, label)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        dataset.eval()

        feature, label = dataset.all()
        prediction = self.layer(feature)
        loss = self.loss_fn(prediction, label)
        return prediction, loss

In [8]:
LEARNING_RATE = 0.00001

In [9]:
BATCH_SIZE = 2

In [10]:
EPOCHS = 1000

In [11]:
dataset = Dataset(BATCH_SIZE)
layer = Linear(2, 1)
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)
model = NNModel(layer, loss_fn, optimizer)

In [12]:
model.train(dataset, EPOCHS)

In [13]:
prediction, loss = model.test(dataset)
print(f'prediction:\t{prediction}\nloss:\t{loss}')

prediction:	Tensor([[163.52327795]])
loss:	Tensor(2.1807080003283335)
